In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.layers import LayerNormalization, Dense, Dropout, Input, Embedding
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import AdamW



# Positional Encoding Function
def positional_encoding(seq_len, d_model):
    positions = np.arange(seq_len)[:, np.newaxis]
    div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))

    pos_encoding = np.zeros((seq_len, d_model))
    pos_encoding[:, 0::2] = np.sin(positions * div_term)
    pos_encoding[:, 1::2] = np.cos(positions * div_term)

    return tf.constant(pos_encoding, dtype=tf.float32)


# Transformer Encoder Block
class TransformerEncoderBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout_rate=0.2):
        super(TransformerEncoderBlock, self).__init__()
        self.attention = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.norm1 = LayerNormalization(epsilon=1e-6)
        self.norm2 = LayerNormalization(epsilon=1e-6)
        self.dense1 = Dense(ff_dim, activation="relu")
        self.dense2 = Dense(embed_dim)
        self.dropout1 = Dropout(dropout_rate)
        self.dropout2 = Dropout(dropout_rate)

    def call(self, inputs, training=False):
        attn_output = self.attention(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.norm1(inputs + attn_output)

        dense_output = self.dense1(out1)
        dense_output = self.dense2(dense_output)
        dense_output = self.dropout2(dense_output, training=training)
        return self.norm2(out1 + dense_output)

In [ ]:
# Model Parameters
embed_dim = 256   # Increased embedding size
num_heads = 8     # More attention heads
ff_dim = 1024     # Larger feed-forward network
input_length = 100
vocab_size = 5000



In [ ]:
# Input Layer
inputs = Input(shape=(input_length,), dtype=tf.int32)
embedding = Embedding(input_dim=vocab_size, output_dim=embed_dim, input_length=input_length)(inputs)



/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
# Add Positional Encoding
pos_encoding = positional_encoding(input_length, embed_dim)
embedding = embedding + pos_encoding



In [ ]:
# Stack Multiple Transformer Blocks
x = embedding
for _ in range(6):  # More layers for deeper learning
    x = TransformerEncoderBlock(embed_dim, num_heads, ff_dim)(x)



In [ ]:
# Dropout and Final Dense Layer
x = Dropout(0.3)(x)  # Increased dropout for regularization
outputs = Dense(vocab_size, activation="softmax")(x)



In [ ]:
# Define Model
transformer = Model(inputs=inputs, outputs=outputs, name="optimized_transformer")



In [ ]:
# Learning Rate Scheduler
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(initial_learning_rate=1e-3, decay_steps=10000, decay_rate=0.9)



In [ ]:
# Compile Model
transformer.compile(optimizer=AdamW(learning_rate=lr_schedule), loss="sparse_categorical_crossentropy", metrics=["accuracy"])



In [ ]:
# Show Model Summary
transformer.summary()

Model: "optimized_transformer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 100, 256)       │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ add (Add)                       │ (None, 100, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder_block       │ (None, 100, 256)       │     2,630,144 │
│ (TransformerEncoderBlock)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder_block_1     │ (None, 100, 256)       │     2,630,144 │
│ (TransformerEncoderBlock)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder_block_2     │ (None, 100, 256)       │     2,630,144 │
│ (TransformerEncoderBlock)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder_block_3     │ (None, 100, 256)       │     2,630,144 │
│ (TransformerEncoderBlock)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder_block_4     │ (None, 100, 256)       │     2,630,144 │
│ (TransformerEncoderBlock)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder_block_5     │ (None, 100, 256)       │     2,630,144 │
│ (TransformerEncoderBlock)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_18 (Dropout)            │ (None, 100, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 100, 5000)      │     1,285,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 18,345,864 (69.98 MB)

 Trainable params: 18,345,864 (69.98 MB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
!pip install keras-tuner --quiet

import tensorflow as tf
import keras_tuner as kt
import numpy as np
from tensorflow.keras.layers import Input, Dense, Embedding, Dropout, LayerNormalization, GlobalAveragePooling1D, MultiHeadAttention
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from sklearn.metrics import classification_report, precision_score, recall_score
from tensorflow.keras.callbacks import EarlyStopping

# Set seed for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Custom Transformer Encoder Block
class TransformerEncoderBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout_rate=0.1):
        super(TransformerEncoderBlock, self).__init__()
        self.attention = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.norm1 = LayerNormalization(epsilon=1e-6)
        self.norm2 = LayerNormalization(epsilon=1e-6)
        self.dense1 = Dense(ff_dim, activation="relu")
        self.dense2 = Dense(embed_dim)
        self.dropout1 = Dropout(dropout_rate)
        self.dropout2 = Dropout(dropout_rate)

    def call(self, inputs, training=False):
        attn_output = self.attention(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.norm1(inputs + attn_output)

        dense_output = self.dense1(out1)
        dense_output = self.dense2(dense_output)
        dense_output = self.dropout2(dense_output, training=training)
        return self.norm2(out1 + dense_output)

# Synthetic data generator
def generate_synthetic_sequences(n_samples=1000, seq_len=100, vocab_size=5000):
    X = np.zeros((n_samples, seq_len), dtype=np.int32)
    y = np.zeros(n_samples, dtype=np.int32)

    for i in range(n_samples):
        label = np.random.randint(0, 2)
        y[i] = label
        if label == 0:
            X[i] = np.random.randint(0, vocab_size // 2, size=seq_len)
        else:
            X[i] = np.random.randint(vocab_size // 2, vocab_size, size=seq_len)
    return X, y

# Build model with tunable parameters
def build_transformer_model(hp):
    embed_dim = hp.Choice('embed_dim', values=[128])
    num_heads = hp.Choice('num_heads', values=[2, 4])
    ff_dim = hp.Choice('ff_dim', values=[256])
    dropout_rate = hp.Float('dropout_rate', min_value=0.1, max_value=0.3, step=0.1)

    inputs = Input(shape=(100,))
    embedding = Embedding(input_dim=5000, output_dim=embed_dim)(inputs)

    x = TransformerEncoderBlock(embed_dim, num_heads, ff_dim, dropout_rate)(embedding)
    x = GlobalAveragePooling1D()(x)
    x = Dropout(dropout_rate)(x)

    outputs = Dense(2, activation="softmax")(x)

    model = Model(inputs=inputs, outputs=outputs)

    model.compile(
        optimizer=Adam(learning_rate=hp.Choice('learning_rate', values=[1e-3, 1e-4])),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

    return model

# Generate training/validation data
x_train, y_train = generate_synthetic_sequences(1000)
x_val, y_val = generate_synthetic_sequences(200)

# Tuner using RandomSearch with fewer trials
tuner = kt.RandomSearch(
    build_transformer_model,
    objective="val_accuracy",
    max_trials=3,  # ⬅️ Reduced number of trials
    executions_per_trial=1,
    directory="transformer_tuning",
    project_name="transformer_light"
)

# Run the tuner
tuner.search(x_train, y_train, epochs=5, validation_data=(x_val, y_val), verbose=1)

# Retrieve best model
best_hps = tuner.get_best_hyperparameters(1)[0]
print(f"\n✅ Best Hyperparameters: {best_hps.values}\n")

best_model = tuner.hypermodel.build(best_hps)

# Train best model
history = best_model.fit(
    x_train, y_train,
    validation_data=(x_val, y_val),
    epochs=15,
    batch_size=64,
    callbacks=[EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)],
    verbose=1
)

# Final evaluation
test_loss, test_acc = best_model.evaluate(x_val, y_val)
print(f"\n✅ Test Accuracy: {test_acc:.4f}")

# Metrics
y_pred = np.argmax(best_model.predict(x_val), axis=1)
precision = precision_score(y_val, y_pred)
recall = recall_score(y_val, y_pred)

print(f"✅ Precision: {precision:.4f}")
print(f"✅ Recall: {recall:.4f}")
print("\n📊 Classification Report:\n", classification_report(y_val, y_pred))


Trial 3 Complete [00h 00m 15s]
val_accuracy: 1.0

Best val_accuracy So Far: 1.0
Total elapsed time: 00h 00m 53s

✅ Best Hyperparameters: {'embed_dim': 128, 'num_heads': 4, 'ff_dim': 256, 'dropout_rate': 0.2, 'learning_rate': 0.0001}

Epoch 1/15
16/16 ━━━━━━━━━━━━━━━━━━━━ 14s 407ms/step - accuracy: 0.7038 - loss: 0.6091 - val_accuracy: 1.0000 - val_loss: 0.2782
Epoch 2/15
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - accuracy: 1.0000 - loss: 0.2055 - val_accuracy: 1.0000 - val_loss: 0.0497
Epoch 3/15
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - accuracy: 1.0000 - loss: 0.0332 - val_accuracy: 1.0000 - val_loss: 0.0072
Epoch 4/15
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 1.0000 - loss: 0.0059 - val_accuracy: 1.0000 - val_loss: 0.0024
Epoch 5/15
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 1.0000 - loss: 0.0025 - val_accuracy: 1.0000 - val_loss: 0.0014
Epoch 6/15
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - accuracy: 1.0000 - loss: 0.0016 - val_accuracy: 1.0000 - val_loss: 0.0011
Ep

In [ ]:
import tensorflow as tf
import numpy as np
from sklearn.metrics import classification_report, precision_score, recall_score, f1_score
from tensorflow.keras.layers import Input, Dense, Embedding, Dropout, LayerNormalization, GlobalAveragePooling1D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import MultiHeadAttention

# Set random seeds
np.random.seed(42)
tf.random.set_seed(42)

# Synthetic Sequence Generator
def generate_synthetic_sequences(n_samples=1000, seq_len=100, vocab_size=5000):
    X = np.zeros((n_samples, seq_len), dtype=np.int32)
    y = np.zeros(n_samples, dtype=np.int32)
    for i in range(n_samples):
        label = np.random.randint(0, 2)
        y[i] = label
        if label == 0:
            X[i] = np.random.randint(0, vocab_size // 2, size=seq_len)
        else:
            X[i] = np.random.randint(vocab_size // 2, vocab_size, size=seq_len)
    return X, y

# Positional Encoding
def positional_encoding(seq_len, d_model):
    angles = np.arange(seq_len)[:, np.newaxis] / np.power(10000, (2 * (np.arange(d_model)[np.newaxis, :] // 2)) / np.float32(d_model))
    pos_encoding = np.zeros((seq_len, d_model))
    pos_encoding[:, 0::2] = np.sin(angles[:, 0::2])
    pos_encoding[:, 1::2] = np.cos(angles[:, 1::2])
    return tf.constant(pos_encoding, dtype=tf.float32)

# Transformer Encoder Block
class TransformerEncoderBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout_rate=0.3):
        super().__init__()
        self.attn = MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.norm1 = LayerNormalization()
        self.norm2 = LayerNormalization()
        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim)
        ])
        self.dropout1 = Dropout(dropout_rate)
        self.dropout2 = Dropout(dropout_rate)

    def call(self, inputs, training=False):
        attn_output = self.attn(inputs, inputs)
        out1 = self.norm1(inputs + self.dropout1(attn_output, training=training))
        ffn_output = self.ffn(out1)
        return self.norm2(out1 + self.dropout2(ffn_output, training=training))

# Model Architecture
def build_transformer_model(seq_len=100, vocab_size=5000, embed_dim=128, num_heads=4, ff_dim=512):
    inputs = Input(shape=(seq_len,))
    x = Embedding(input_dim=vocab_size, output_dim=embed_dim)(inputs)
    x = x + positional_encoding(seq_len, embed_dim)

    for _ in range(2):  # Two transformer blocks
        x = TransformerEncoderBlock(embed_dim, num_heads, ff_dim)(x)

    x = GlobalAveragePooling1D()(x)
    x = Dropout(0.3)(x)
    outputs = Dense(2, activation='softmax')(x)

    model = Model(inputs, outputs)
    model.compile(optimizer=Adam(1e-4), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

# Generate Data
x_train, y_train = generate_synthetic_sequences(1000)
x_val, y_val = generate_synthetic_sequences(200)

# Build and Train Model
model = build_transformer_model()
early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)
history = model.fit(x_train, y_train, validation_data=(x_val, y_val), epochs=30, batch_size=64, callbacks=[early_stop], verbose=1)

# Evaluate Model
test_loss, test_acc = model.evaluate(x_val, y_val)
print(f"\n✅ Test Accuracy: {test_acc:.4f}")

# Predict and Print Metrics
y_pred = np.argmax(model.predict(x_val), axis=1)
precision = precision_score(y_val, y_pred, zero_division=0)
recall = recall_score(y_val, y_pred, zero_division=0)
f1 = f1_score(y_val, y_pred, zero_division=0)

print(f"✅ Precision: {precision:.4f}")
print(f"✅ Recall:    {recall:.4f}")
print(f"✅ F1 Score:  {f1:.4f}")
print("\n📊 Classification Report:\n", classification_report(y_val, y_pred, zero_division=0))


Epoch 1/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 23s 654ms/step - accuracy: 0.5172 - loss: 0.8268 - val_accuracy: 0.5100 - val_loss: 0.7314
Epoch 2/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.5018 - loss: 0.7804 - val_accuracy: 0.5100 - val_loss: 0.6883
Epoch 3/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.5605 - loss: 0.7026 - val_accuracy: 0.5100 - val_loss: 0.6588
Epoch 4/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.5605 - loss: 0.6949 - val_accuracy: 0.5100 - val_loss: 0.6134
Epoch 5/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.6595 - loss: 0.6168 - val_accuracy: 0.5100 - val_loss: 0.5775
Epoch 6/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 39ms/step - accuracy: 0.7218 - loss: 0.5743 - val_accuracy: 1.0000 - val_loss: 0.4743
Epoch 7/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - accuracy: 0.9007 - loss: 0.4697 - val_accuracy: 1.0000 - val_loss: 0.3081
Epoch 8/30
16/16 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step - accuracy: 0.9924 - loss: 0.2884 - val_accuracy: 1.0000 -

In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.layers import LayerNormalization, Dense, Dropout, Input, Embedding
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import AdamW

# Positional Encoding
def positional_encoding(seq_len, d_model):
    positions = np.arange(seq_len)[:, np.newaxis]
    div_term = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
    pos_encoding = np.zeros((seq_len, d_model))
    pos_encoding[:, 0::2] = np.sin(positions * div_term)
    pos_encoding[:, 1::2] = np.cos(positions * div_term)
    return tf.constant(pos_encoding, dtype=tf.float32)

# Transformer Encoder Block
class TransformerEncoderBlock(tf.keras.layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout_rate=0.3):
        super(TransformerEncoderBlock, self).__init__()
        self.attention = tf.keras.layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.norm1 = LayerNormalization(epsilon=1e-6)
        self.norm2 = LayerNormalization(epsilon=1e-6)
        self.dense1 = Dense(ff_dim, activation="relu")
        self.dense2 = Dense(embed_dim)
        self.dropout1 = Dropout(dropout_rate)
        self.dropout2 = Dropout(dropout_rate)

    def call(self, inputs, training=False):
        attn_output = self.attention(inputs, inputs)
        attn_output = self.dropout1(attn_output, training=training)
        out1 = self.norm1(inputs + attn_output)

        dense_output = self.dense1(out1)
        dense_output = self.dense2(dense_output)
        dense_output = self.dropout2(dense_output, training=training)
        return self.norm2(out1 + dense_output)

# Model Parameters
embed_dim = 128
num_heads = 4
ff_dim = 512
input_length = 100
vocab_size = 5000
num_layers = 4

# Input Layer
inputs = Input(shape=(input_length,), dtype=tf.int32)
embedding = Embedding(input_dim=vocab_size, output_dim=embed_dim, input_length=input_length)(inputs)

# Add Positional Encoding
pos_encoding = positional_encoding(input_length, embed_dim)
embedding = embedding + pos_encoding

# Stack Transformer Blocks
x = embedding
for _ in range(num_layers):
    x = TransformerEncoderBlock(embed_dim, num_heads, ff_dim)(x)

# Dropout and Final Dense Layer
x = Dropout(0.3)(x)
outputs = Dense(vocab_size, activation="softmax")(x)

# Define Model
transformer = Model(inputs=inputs, outputs=outputs, name="optimized_transformer")

# Learning Rate Scheduler
lr_schedule = tf.keras.optimizers.schedules.ExponentialDecay(
    initial_learning_rate=1e-4, decay_steps=10000, decay_rate=0.9
)

# Compile Model
transformer.compile(optimizer=AdamW(learning_rate=lr_schedule),
                    loss="sparse_categorical_crossentropy",
                    metrics=["accuracy"])

# Model Summary
transformer.summary()


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "optimized_transformer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 100)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 100, 128)       │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ add (Add)                       │ (None, 100, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder_block       │ (None, 100, 128)       │       396,032 │
│ (TransformerEncoderBlock)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder_block_1     │ (None, 100, 128)       │       396,032 │
│ (TransformerEncoderBlock)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder_block_2     │ (None, 100, 128)       │       396,032 │
│ (TransformerEncoderBlock)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_encoder_block_3     │ (None, 100, 128)       │       396,032 │
│ (TransformerEncoderBlock)       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ (None, 100, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 100, 5000)      │       645,000 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,869,128 (10.94 MB)

 Trainable params: 2,869,128 (10.94 MB)

 Non-trainable params: 0 (0.00 B)